# NLP — Text Classification from Scratch to Fine-tuning

Progressive workflow: bag-of-words → TF-IDF + linear → embeddings.
Always establish a strong linear baseline before reaching for transformers.

## TF-IDF + Logistic Regression baseline

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 4 well-separated categories
CATS = ["sci.med", "sci.space", "rec.sport.baseball", "talk.politics.guns"]
train = fetch_20newsgroups(subset="train", categories=CATS,
                            remove=("headers","footers","quotes"))
test  = fetch_20newsgroups(subset="test",  categories=CATS,
                            remove=("headers","footers","quotes"))

pipe = Pipeline([
    ("vec", TfidfVectorizer(max_features=30_000, ngram_range=(1,2))),
    ("clf", LogisticRegression(C=5.0, max_iter=1000, random_state=42)),
])
pipe.fit(train.data, train.target)
preds = pipe.predict(test.data)
print(classification_report(test.target, preds, target_names=CATS))

## Top features per class — model interpretability

In [ ]:
import numpy as np

vocab = pipe.named_steps["vec"].get_feature_names_out()
coef  = pipe.named_steps["clf"].coef_

print("Top 10 features per class:\n")
for i, cat in enumerate(CATS):
    top10 = np.argsort(coef[i])[-10:][::-1]
    print(f"{cat}:")
    print("  " + ", ".join(vocab[top10]))
    print()

## Sentence embeddings for semantic similarity

In [ ]:
# Demonstrates the concept with TF-IDF cosine similarity
# In production: replace with sentence-transformers embeddings
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "machine learning for fraud detection in banking",
    "credit card fraud prevention with neural networks",
    "natural language processing for sentiment analysis",
    "deep learning for customer review classification",
    "random forest for churn prediction",
]
query = "detecting financial fraud with ML"

vec = TfidfVectorizer().fit(docs + [query])
doc_vecs   = vec.transform(docs)
query_vec  = vec.transform([query])
sims = cosine_similarity(query_vec, doc_vecs)[0]

print(f"Query: '{query}'\n")
print("Similarity scores:")
for doc, sim in sorted(zip(docs, sims), key=lambda x: -x[1]):
    print(f"  {sim:.3f}  {doc}")
print()
print("Note: in production use sentence-transformers for semantic similarity")
print("      (pip install sentence-transformers)")